In [1]:
#this scripts uses calculated thermo cac equilibrium data to train ML models to predict D_max for different compositions. This uses CBFV as additional features.


In [2]:
#Import necessary libraries
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from CBFV import composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import re
from ast import literal_eval
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.core.arm import Arm

c:\Users\Chris\Documents\ML_Project\ML_GFA_env\Lib\site-packages\CBFV\composition.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
#according to the single tree notebook the best temperature ranges to utalize are: 2200, 1700, 2450, 2100, 2400, 1900, 1850
#according to the same notebook the NF or phase fraction features are ineffective at predicting D_max and thus will not be used

In [10]:
#pull training data
train_CALPHAD_df = pd.read_csv(r"Data/F(Composition)_Data/CALPHAD_alloys_train_opt.csv")

#pull test data
test_CALPHAD_df = pd.read_csv(r"Data/F(Composition)_Data/CALPHAD_alloys_test.csv")


train_CALPHAD_df.head()

,alloy_string,DF_AG2CA_T0C,DF_AG2CA_T1000C,DF_AG2CA_T100C,DF_AG2CA_T1050C,DF_AG2CA_T1100C,DF_AG2CA_T1150C,DF_AG2CA_T1200C,DF_AG2CA_T1250C,DF_AG2CA_T1300C,...,NF_ZRSI_T50C,NF_ZRSI_T550C,NF_ZRSI_T600C,NF_ZRSI_T650C,NF_ZRSI_T700C,NF_ZRSI_T750C,NF_ZRSI_T800C,NF_ZRSI_T850C,NF_ZRSI_T900C,NF_ZRSI_T950C
0,B22.00Co4.00Fe68.00Y6.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Al10.00Ce60.00Cu20.00Ni10.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
#create the CBFV features for the training and test data
#start by pulling the formula column
train_formula_df = pd.DataFrame({'formula': train_CALPHAD_df['alloy_string']})
test_formula_df = pd.DataFrame({'formula': test_CALPHAD_df['alloy_string']})

#add a target column for CBFV api
train_formula_df['target'] = 0
test_formula_df['target'] = 0

#use the CBFV api to create the features
train_CBFV_df, _, train_formulae, skipped_train = composition.generate_features(train_formula_df, elem_prop='magpie')
test_CBFV_df, _, test_formulae, skipped_test = composition.generate_features(test_formula_df, elem_prop='magpie')


print(f"CBFV train features shape: {train_CBFV_df.shape}, test features shape: {test_CBFV_df.shape}")
print(f"Number of skipped formulas: {len(skipped_train)}, {len(skipped_test)}")
train_CALPHAD_df.head()


Processing Input Data: 100%|██████████| 882/882 [00:00<00:00, 27562.03it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 882/882 [00:00<00:00, 24163.77it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 98/98 [00:00<00:00, 12252.71it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 98/98 [00:00<00:00, 16337.11it/s]

	Creating Pandas Objects...
CBFV train features shape: (882, 132), test features shape: (98, 132)
Number of skipped formulas: 0, 0


,alloy_string,DF_AG2CA_T0C,DF_AG2CA_T1000C,DF_AG2CA_T100C,DF_AG2CA_T1050C,DF_AG2CA_T1100C,DF_AG2CA_T1150C,DF_AG2CA_T1200C,DF_AG2CA_T1250C,DF_AG2CA_T1300C,...,NF_ZRSI_T50C,NF_ZRSI_T550C,NF_ZRSI_T600C,NF_ZRSI_T650C,NF_ZRSI_T700C,NF_ZRSI_T750C,NF_ZRSI_T800C,NF_ZRSI_T850C,NF_ZRSI_T900C,NF_ZRSI_T950C
0,B22.00Co4.00Fe68.00Y6.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Al10.00Ce60.00Cu20.00Ni10.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
#filter the CALPHAD data to include only driving forces at the temepratures of interests
# Define the temperature ranges of interest
temperature_range = [2200, 1700, 2450, 2100, 2400, 1900, 1850]
temperature_range_str = [str(temp) for temp in temperature_range]

# Filter the columns to include only those with the specified temperature ranges and Df
filtered_CALPHAD_cols = [col for col in train_CALPHAD_df.columns if any(temp in col for temp in temperature_range_str) and 'DF' in col]

print(f"Number of filtered CALPHAD columns: {len(filtered_CALPHAD_cols)}")

Number of filtered CALPHAD columns: 5628


In [18]:
#combine the CBFV features with the filtered CALPHAD features for training and test data
X_train = pd.concat([train_CBFV_df, train_CALPHAD_df[filtered_CALPHAD_cols]], axis=1)
X_test = pd.concat([test_CBFV_df, test_CALPHAD_df[filtered_CALPHAD_cols]], axis=1)

print(f"Combined train features shape: {X_train.shape}, Combined test features shape: {X_test.shape}")


Combined train features shape: (882, 5760), Combined test features shape: (98, 5760)
